# Sentiment Analysis

In [12]:
# Import libraries for numerical computation, data manipulation, and deep learning.
import numpy as np
import pandas as pd
import torch

# Import the Hugging Face classes used to initialize pretrained tokenizers and sequence-classification models.
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Specify the pretrained financial sentiment-analysis models to be evaluated.
# Dictionary keys provide internal identifiers, while values specify the corresponding model repositories on Hugging Face.
MODEL_NAMES = {
    "financial_roberta": "soleimanian/financial-roberta-large-sentiment",
    "finbert": "ProsusAI/finbert",
}

# Set the computation device to the CPU to ensure model inference does not depend on the availability of a compatible GPU.
device = torch.device("cpu")

# Initialize dictionaries to store the models, tokenizers, class labels, and maximum text chunk sizes associated with each model.
models = {}
tokenizers = {}
model_labels = {}
chunk_sizes = {}

# Define the sentiment classes required for subsequent comparative analysis.
EXPECTED_LABELS = {"negative", "neutral", "positive"}

# Load each pretrained model and validate its classification configuration.
for model_key, model_name in MODEL_NAMES.items():

    # Report the model currently being initialized.
    print(f"Loading {model_key}: {model_name}")

    # Load the tokenizer and pretrained sequence-classification model associated with the specified Hugging Face repository.
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name)

    # Move the model to the selected computation device and enable evaluation mode to disable training-specific behavior during inference.
    model.to(device)
    model.eval()

    # Retrieve the model's output labels in their original classification-index order, converting them to lowercase for consistent processing.
    labels = [
        str(model.config.id2label[i]).lower()
        for i in range(model.config.num_labels)
    ]

    # Define a mapping to standardize common sentiment abbreviations and generic class identifiers into the required sentiment categories.
    label_mapping = {
        "label_0": "negative",
        "label_1": "neutral",
        "label_2": "positive",
        "neg": "negative",
        "neu": "neutral",
        "pos": "positive",
    }

    # Standardize the retrieved labels while preserving their original order.
    labels = [label_mapping.get(label, label) for label in labels]

    # Verify that the model contains exactly the three expected sentiment classes; terminate execution if the classification configuration differs.
    if len(labels) != 3 or set(labels) != EXPECTED_LABELS:
        raise ValueError(
            f"{model_key} does not have the expected three classes. "
            f"Found labels: {labels}. Check the model card."
        )

    # Retrieve the model's configured positional limit, using 512 tokens as a fallback when the configuration does not specify this attribute.
    configured_limit = getattr(
        model.config, "max_position_embeddings", 512
    )

    # Retrieve the tokenizer's advertised maximum input length.
    tokenizer_limit = tokenizer.model_max_length

    # Replace extremely large tokenizer limits, which may be sentinel values, with the model's configured positional limit.
    if tokenizer_limit > 100_000:
        tokenizer_limit = configured_limit

    # Determine the effective maximum input length from the tokenizer and model configuration limits.
    max_length = min(tokenizer_limit, configured_limit)

    # Calculate the maximum number of text tokens per chunk by subtracting the number of special tokens added for a single input sequence.
    chunk_size = (
        max_length
        - tokenizer.num_special_tokens_to_add(pair=False)
    )

    # Store the initialized model, tokenizer, standardized labels, and calculated chunk size for subsequent sentiment-analysis procedures.
    models[model_key] = model
    tokenizers[model_key] = tokenizer
    model_labels[model_key] = labels
    chunk_sizes[model_key] = chunk_size

    # Report the classification labels and input-length parameters to facilitate verification of the model initialization procedure.
    print("Labels:", labels)
    print("Maximum length:", max_length)
    print("Chunk size:", chunk_size)
    print("-" * 50)

# Confirm that both models have been loaded and passed the label validation.
print("Both models loaded successfully.")


Loading financial_roberta: soleimanian/financial-roberta-large-sentiment
Labels: ['negative', 'neutral', 'positive']
Maximum length: 514
Chunk size: 512
--------------------------------------------------
Loading finbert: ProsusAI/finbert
Labels: ['positive', 'negative', 'neutral']
Maximum length: 512
Chunk size: 510
--------------------------------------------------
Both models loaded successfully.


In [13]:
def predict_sentiment(text, model_key, batch_size=16):

    # Handle missing or empty paragraphs
    if pd.isna(text) or not str(text).strip():
        return {
            "sentiment": None,
            "positive_prob": np.nan,
            "neutral_prob": np.nan,
            "negative_prob": np.nan,
            "n_chunks": 0,
        }

    tokenizer = tokenizers[model_key]
    model = models[model_key]
    labels = model_labels[model_key]
    chunk_size = chunk_sizes[model_key]

    # Tokenize without truncating the paragraph
    token_ids = tokenizer(
        str(text),
        add_special_tokens=False,
        truncation=False,
    )["input_ids"]

    # Split into model-compatible chunks
    chunks = [
        token_ids[i:i + chunk_size]
        for i in range(0, len(token_ids), chunk_size)
    ]

    if not chunks:
        return {
            "sentiment": None,
            "positive_prob": np.nan,
            "neutral_prob": np.nan,
            "negative_prob": np.nan,
            "n_chunks": 0,
        }

    all_probabilities = []

    for start in range(0, len(chunks), batch_size):

        batch_chunks = chunks[start:start + batch_size]

        encoded_chunks = [
            tokenizer.prepare_for_model(
                chunk,
                add_special_tokens=True,
                truncation=False,
                return_attention_mask=True,
            )
            for chunk in batch_chunks
        ]

        batch_inputs = tokenizer.pad(
            encoded_chunks,
            padding=True,
            return_tensors="pt",
        )

        with torch.no_grad():
            logits = model(**batch_inputs).logits
            probabilities = torch.softmax(logits, dim=1)

        all_probabilities.extend(probabilities.cpu().numpy())

    # Average probabilities across chunks
    mean_probs = np.mean(all_probabilities, axis=0)

    probs = {
        label: float(mean_probs[i])
        for i, label in enumerate(labels)
    }

    sentiment = max(probs, key=probs.get)

    return {
        "sentiment": sentiment,
        "positive_prob": probs["positive"],
        "neutral_prob": probs["neutral"],
        "negative_prob": probs["negative"],
        "n_chunks": len(chunks),
    }

In [14]:
data_adidas = pd.read_excel("../data/input/adidas.xlsx", sheet_name="Sheet1")
data_puma = pd.read_excel("../data/input/puma.xlsx", sheet_name="Sheet1")
data_asics = pd.read_excel("../data/input/asics.xlsx", sheet_name="Sheet1")
data_mizuno = pd.read_excel("../data/input/mizuno.xlsx", sheet_name="Sheet1")

datasets = {
    "adidas": data_adidas,
    "puma": data_puma,
    "asics": data_asics,
    "mizuno": data_mizuno,
}

# Check token lenghts before sentiment analysis

# Create a dictionary to store the token-length results.
token_length_results = {}

# Check every company dataset using both model tokenizers.
for company, data in datasets.items():

    # Create a copy to preserve the original dataset.
    df = data.copy()

    # Calculate token counts separately for each model.
    for model_key, tokenizer in tokenizers.items():

        # Count tokens in each paragraph without adding special tokens.
        token_column = f"{model_key}_token_count"

        df[token_column] = df["Text"].apply(
            lambda text: len(
                tokenizer.encode(
                    str(text),
                    add_special_tokens=False
                )
            ) if pd.notna(text) else 0
        )

        # Compare paragraph lengths with the calculated chunk size.
        limit = chunk_sizes[model_key]
        too_long = df[token_column] > limit

        # Print summary statistics for the current company and model.
        print(f"\nCompany: {company}")
        print(f"Model: {model_key}")
        print(f"Number of paragraphs: {len(df)}")
        print(f"Longest paragraph: {df[token_column].max()} tokens")
        print(f"Average paragraph length: {df[token_column].mean():.2f} tokens")
        print(f"Paragraphs exceeding {limit} tokens: {too_long.sum()}")

    # Store the dataset with its token counts.
    token_length_results[company] = df

# Save all token-length results to separate Excel files for inspection.
for company, df in token_length_results.items():
    df.to_excel(
        f"../data/processed/{company}_token_counts.xlsx",
        index=False
    )

summary_path = "../data/processed/token_length_summary.txt"

# Open the text file for writing.
with open(summary_path, "w", encoding="utf-8") as file:

    # Write the report title.
    file.write("TOKEN LENGTH ANALYSIS SUMMARY\n")
    file.write("=" * 50 + "\n\n")

    # Summarize the results for each company and model.
    for company, df in token_length_results.items():

        file.write(f"Company: {company}\n")
        file.write(f"Number of paragraphs: {len(df)}\n")

        for model_key in tokenizers:

            # Retrieve the token-count column and model-specific limit.
            column = f"{model_key}_token_count"
            limit = chunk_sizes[model_key]

            # Calculate summary statistics.
            longest = df[column].max()
            average = df[column].mean()
            exceeding = (df[column] > limit).sum()

            # Write the results to the report.
            file.write(f"Model: {model_key}\n")
            file.write(f"Maximum text chunk size: {limit} tokens\n")
            file.write(f"Longest paragraph: {longest} tokens\n")
            file.write(f"Average paragraph length: {average:.2f} tokens\n")
            file.write(
                f"Paragraphs exceeding the limit: {exceeding}\n"
            )
            file.write("-" * 40 + "\n")

        file.write("\n")

#print(f"Summary saved to: {summary_path}")

print("\nToken-length checks completed.")



Company: adidas
Model: financial_roberta
Number of paragraphs: 219
Longest paragraph: 315 tokens
Average paragraph length: 81.88 tokens
Paragraphs exceeding 512 tokens: 0

Company: adidas
Model: finbert
Number of paragraphs: 219
Longest paragraph: 318 tokens
Average paragraph length: 80.28 tokens
Paragraphs exceeding 510 tokens: 0

Company: puma
Model: financial_roberta
Number of paragraphs: 105
Longest paragraph: 344 tokens
Average paragraph length: 90.50 tokens
Paragraphs exceeding 512 tokens: 0

Company: puma
Model: finbert
Number of paragraphs: 105
Longest paragraph: 329 tokens
Average paragraph length: 90.58 tokens
Paragraphs exceeding 510 tokens: 0

Company: asics
Model: financial_roberta
Number of paragraphs: 206
Longest paragraph: 374 tokens
Average paragraph length: 76.41 tokens
Paragraphs exceeding 512 tokens: 0

Company: asics
Model: finbert
Number of paragraphs: 206
Longest paragraph: 369 tokens
Average paragraph length: 76.11 tokens
Paragraphs exceeding 510 tokens: 0

Com

Before performing sentiment analysis, the token lengths of all 592 paragraphs were evaluated using the respective tokenizers of Financial RoBERTa and FinBERT. The calculated maximum text capacities were 512 tokens for Financial RoBERTa and 510 tokens for FinBERT, excluding special tokens added during input preparation.

The longest paragraph contained 374 tokens according to the Financial RoBERTa tokenizer and 369 tokens according to the FinBERT tokenizer. No paragraphs exceeded the respective calculated text capacities. Consequently, paragraph splitting was not required on the basis of input length, allowing each paragraph to be retained as a single unit of analysis for subsequent sentiment classification.

In [15]:
results = {}

for company, data in datasets.items():

    df = data.copy()

    # Apply both models to every paragraph
    for model_key in MODEL_NAMES:

        predictions = df["Text"].apply(
            lambda text: predict_sentiment(text, model_key)
        )

        prediction_df = pd.DataFrame(
            predictions.tolist(),
            index=df.index,
        )

        # Add a prefix to identify the model
        prediction_df = prediction_df.add_prefix(
            f"{model_key}_"
        )

        df = pd.concat(
            [df, prediction_df],
            axis=1,
        )

    results[company] = df

    print(f"{company}: {len(df)} text block analyzed")


You're using a RobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


adidas: 219 text block analyzed
puma: 105 text block analyzed
asics: 206 text block analyzed
mizuno: 62 text block analyzed


In [16]:
summary_rows = []

for company, df in results.items():

    for model_key in MODEL_NAMES:

        sentiment_col = f"{model_key}_sentiment"

        proportions = (
            df[sentiment_col]
            .dropna()
            .value_counts(normalize=True)
            .reindex(
                ["negative", "neutral", "positive"],
                fill_value=0,
            )
            .mul(100)
        )

        for sentiment, percentage in proportions.items():

            summary_rows.append({
                "company": company,
                "model": model_key,
                "sentiment": sentiment,
                "percentage": percentage,
            })

sentiment_summary = pd.DataFrame(summary_rows)

comparison = sentiment_summary.pivot_table(
    index=["company", "sentiment"],
    columns="model",
    values="percentage",
)

print(comparison.round(2))

model              financial_roberta  finbert
company sentiment                            
adidas  negative                3.65     2.28
        neutral                39.73    71.23
        positive               56.62    26.48
asics   negative                2.91     1.46
        neutral                32.52    59.22
        positive               64.56    39.32
mizuno  negative                3.23     0.00
        neutral                33.87    59.68
        positive               62.90    40.32
puma    negative               11.43    11.43
        neutral                25.71    21.90
        positive               62.86    66.67


In [18]:
from pathlib import Path

output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)
sentiment_summary = []

for company, df in results.items():

    output_path = output_dir / f"{company}_sentiment.xlsx"

    df.to_excel(output_path, index=False)

    print(f"Saved: {output_path}")

# Convert sentiment summary to a DataFrame
sentiment_summary_df = pd.DataFrame(sentiment_summary)

# Save summary
sentiment_summary_df.to_excel(
    output_dir / "sentiment_summary.xlsx",
    index=False
)

print("All sentiment results saved.")


Saved: ../data/processed/adidas_sentiment.xlsx
Saved: ../data/processed/puma_sentiment.xlsx
Saved: ../data/processed/asics_sentiment.xlsx
Saved: ../data/processed/mizuno_sentiment.xlsx
All sentiment results saved.
